In [2]:
import pandas as pd
import geopandas as gp

## Load Files

In [3]:
# Shapes
la_shapes = gp.read_file("./raw-from-source/LA_2024_01_VTD_SHAPE/LA_2024_01_VTDS.shp")

# Election results
la_elections = pd.read_csv("./la_2024_prim_prec/la_2024_prim_prec.csv")
la_elections["join_id"] = la_elections["COUNTYFP"].astype(str) + la_elections["Precinct"].astype(str)

In [4]:
races = [i for i in list(la_elections.columns) if i not in ['join_id','UNIQUE_ID',
 'COUNTYFP',
 'Parish',
 'Precinct']]

In [5]:
prec_values = pd.read_csv("./raw-from-source/prec_joining_table.csv")

In [38]:
prec_values

,GEOID20,DISSOLVE_ALPHA,UNIQUE_ID
0,220010001-1,220010001-1,Acadia-:-01 01
1,220010001-2,220010001-2,Acadia-:-01 02
2,220010001-3,220010001-3,Acadia-:-01 03
3,220010001-4,220010001-4,Acadia-:-01 04
4,220010001-5,220010001-5,Acadia-:-01 05
...,...,...,...
3827,221270007-1,221270007-1,Winn-:-07 1
3828,22127007-1A,22127007-1A,Winn-:-07 1A
3829,221270007-2,221270007-2,Winn-:-07 2
3830,22127007-2A,22127007-2A,Winn-:-07 2A


In [5]:
elec_merge_data = pd.merge(la_elections, prec_values, left_on = "UNIQUE_ID", right_on = "UNIQUE_ID", how = "left")

In [6]:
data_for_join = elec_merge_data.groupby("DISSOLVE_ALPHA", as_index = False).sum()

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_96938/4166458810.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  data_for_join = elec_merge_data.groupby("DISSOLVE_ALPHA", as_index = False).sum()


In [7]:
data_for_join.drop(["COUNTYFP"], axis = 1, inplace = True)

In [8]:
final_join = gp.GeoDataFrame(pd.merge(la_shapes, data_for_join, left_on = "GEOID20", right_on = "DISSOLVE_ALPHA", how = "outer", indicator = True))

In [9]:
la_shapes.columns

Index(['STATEFP20', 'COUNTYFP20', 'VTDST20', 'GEOID20', 'VTDI20', 'NAME20',
       'NAMELSAD20', 'LSAD20', 'MTFCC20', 'FUNCSTAT20', 'ALAND20', 'AWATER20',
       'INTPTLAT20', 'INTPTLON20', 'UNITNUM', 'UNIT_CODE', 'GLEVEL',
       'UNIT_NAME', 'COUNTY', 'MCD', 'PLACE', 'VTD', 'TRACT', 'BGROUP',
       'BLOCK', 'TOT_POP', 'TOT_WHITE', 'TOT_BLACK', 'TOT_ASIAN', 'TOT_AMIND',
       'TOT_OTHER', 'TOT_HISPAN', 'VAP_TOTAL', 'VAP_WHITE', 'VAP_BLACK',
       'VAP_ASIAN', 'VAP_AMIND', 'VAP_OTHER', 'VAP_HISPAN', 'SHAPE_AREA',
       'SHAPE_LEN', 'geometry'],
      dtype='object')

In [10]:
final_join["_merge"].value_counts()

both          3678
left_only        6
right_only       0
Name: _merge, dtype: int64

In [11]:
pd.set_option('display.max_columns', None)

In [12]:
final_join[final_join["_merge"]=="left_only"]

,STATEFP20,COUNTYFP20,VTDST20,GEOID20,VTDI20,NAME20,NAMELSAD20,LSAD20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,UNITNUM,UNIT_CODE,GLEVEL,UNIT_NAME,COUNTY,MCD,PLACE,VTD,TRACT,BGROUP,BLOCK,TOT_POP,TOT_WHITE,TOT_BLACK,TOT_ASIAN,TOT_AMIND,TOT_OTHER,TOT_HISPAN,VAP_TOTAL,VAP_WHITE,VAP_BLACK,VAP_ASIAN,VAP_AMIND,VAP_OTHER,VAP_HISPAN,SHAPE_AREA,SHAPE_LEN,geometry,DISSOLVE_ALPHA,P24PREDBID,P24PREDELY,P24PREDLOZ,P24PREDLYO,P24PREDPER,P24PREDPHI,P24PREDUYG,P24PREDWIL,P24PRERBIN,P24PRERCHR,P24PRERDES,P24PRERHAL,P24PRERHUT,P24PRERRAM,P24PRERSTU,P24PRERSWI,P24PRERTRU,SC001ADBOW,SC001ADSMI,SC001ARALP,SC001ARLIO,SC001BDBOW,SC001BDLES,SC001CRDEE,SC001CRSMI,SC001ERDUB,SC001ERHIR,SC001ERSIG,SC001GRBAY,SC001GRGAR,SC001GROWE,SC002ADGIL,SC002ADHAR,SC002ARBER,SC002ARCAL,SC002ARPRU,SC002BRKAT,SC002BRLEE,SC003ARENS,SC003ARHAR,SC003ARPHI,SC003ARSTE,SC003ARSWA,SC003BRLUN,SC003BROST,SC004ADDEN,SC004ADNOR,SC004ADTOM,SC004BDDJO,SC004BDJOH,SC004BDRJO,SC004BRBRI,SC004BRBRU,SC004BRLAU,SC004BRPHI,SC004BRQUE,SC005ADCAL,SC005ADPOS,SC005ARFOS,SC005ARKEP,SC005ARRUO,SC005BDCOX,SC005BDJEA,SC005BRCOO,SC005BRMCE,SC006ADMER,SC006ADOLD,SC006ADSTE,SC006BDCOT,SC006BDDAR,SC006BRLIN,SC006BRSTA,SC006CRKRO,SC006CRWIL,SC006DRCRA,SC006DRMES,SC006ERIVE,SC006ERLAZ,SC006FRCAR,SC006FRPOW,SC006HRBIN,SC006HRMAC,SC007BRLYN,SC007BRPAR,SC007BRWAL,SC008BDLEI,SC008BDSER,SC008CRCAN,SC008CRPRO,SC008CRYOR,SC008DRCLA,SC008DRKIR,SC009ADBAK,SC009ADHOS,SC009ARLEO,SC009ARSER,SC009BRBRO,SC009BRDON,SC009BRTHE,SC009FRBER,SC009FRKIR,SC010BRCAL,SC010BRLEB,SC010BRZIM,SC010DRCAP,SC010DRGUI,SC010FRBAU,SC010FRSOU,SC010GRBRA,SC010GRODE,SC011ADMAC,SC011ADWIL,SC011ARHAM,SC011ARSTR,SC011BDAND,SC011BDWOO,SC011BDYOU,SC011CRFER,SC011CRHOT,SC011FRRAY,SC011FRSCU,SC011GRGAL,SC011GRSHI,SC011KRBON,SC011KRCAS,SC011KRTAT,SC012ARCHR,SC012ARVAR,SC012BDCAB,SC012BDEUN,SC012BDSKA,SC012ERBRE,SC012ERFRI,SC013ARKEE,SC013ARLAS,SC013DRBAB,SC013DRCRO,SC013GRAND,SC013GRMOR,SC014ARCHI,SC014ARJER,SC014ARKIT,SC014ARSAV,SC014BDALL,SC014BDNOE,SC016AREDG,SC016ARHAI,SC016CRCGR,SC016CRGRA,SC016CRMCC,SC017ARFAB,SC017ARLEM,SC017DRHUV,SC017DRPEY,SC017FRBIL,SC017FRHAR,SC018ADBEN,SC018ADPAU,SC018ARBEN,SC018ARFOR,SC018CRALO,SC018CRCAR,SC018CRFOS,SC018DRLAM,SC018DRSIM,SC018FRAYD,SC018FRNEF,SC019ARDUE,SC019ARMON,SC019ERJAM,SC019ERSUM,SC020BRBER,SC020BRWHI,SC020DRLEC,SC020DRTHI,SC020ERTRO,SC020ERWAL,SC020FRBAZ,SC020FRENG,SC021ARGUL,SC021ARHOW,SC021BDCON,SC021BDDAV,SC021BDJOH,SC021BDSIN,SC021BDWIL,SC021FRCAF,SC021FRPUR,SC022ARBRE,SC022ARROM,SC023ADCLE,SC023ADCON,SC023ADGRE,SC023ARRIC,SC023ARSKI,SC023BDGER,SC023BDMEN,SC023BDSWE,SC023BRANG,SC023BRBEN,SC023BRCAM,SC023BRESC,SC023CRLIN,SC023CRSUP,SC023DRCAM,SC023DRNOR,SC023FREAT,SC023FRIST,SC023JRBUC,SC023JRPER,SC024DRKIS,SC024DRREE,SC025ADSAN,SC025ADSUT,SC026ADWAR,SC026ADWHI,SC026BRHAR,SC026BRNAQ,SC026CRHOO,SC026CRSAV,SC026HRCRA,SC026HRTOT,SC028ADMAR,SC028ADOBR,SC028BDMCK,SC028BDWIL,SC028FRCAV,SC028FRJOH,SC029BDDOG,SC029BDJOR,SC029BDLAN,SC029BDRIC,SC031CREDW,SC031CRPRI,SC031GRDAV,SC031GRWRI,SC032ARHAR,SC032ARMON,SC032CRDEL,SC032CRPHI,SC032GRAVE,SC032GRPAN,SC032HRHAR,SC032HRTAR,SC033FRDAV,SC033FRSTA,SC034BDFON,SC034BDGUI,SC034BDSMI,SC035ARBRO,SC035ARHIN,SC035FRNUN,SC035FRWHE,SC035HRBRO,SC035HRCOO,SC035HRTAR,SC036CRMEE,SC036CRSTI,SC036DRLOM,SC036DRWYC,SC037DRBAI,SC037DRBAR,SC037FRMCI,SC037FRSMI,SC038FRAVA,SC038FRROG,SC039CRERS,SC039CRSAV,SC040BDCLA,SC040BDRIC,SC044ADBRO,SC044ADMIC,SC044BDDAL,SC044BDEDM,SC045ADBER,SC045ADCLA,SC046ADDES,SC046ADGAL,SC050BDCON,SC050BDPAR,SC051BDCAS,SC051BDTRI,SC056BDDAV,SC056BDPRI,SC057BDCOO,SC057BDGAI,SC057BDSOR,SC058ADBAN,SC058ADDUM,SC058ADFRA,SC058ADROU,SC059BDCHR,SC059BDHER,SC060BDGRA,SC060BDMOR,SC061ADCON,SC061ADMAR,SC061BDBEL,SC061BDSMI,SC061BDYAN,SC062BDBAN,SC062BDKEN,SC062BDLOV,SC063ADMIL,SC063ADWRI,SC063BDBLA,SC063BDJOH,SC065ADADK,SC065ADWAM,SC066ADLEG,SC066ADMAT,SC066ADPRI,SC066BDHOL,SC066BDTAY,SC067ADCUL,SC067ADHOL,SC067BDCAS,SC067BDLEW,SC068ADDAV,SC068ADHAR,SC068ADRUS,SC068BDADA,SC068BDSCH,SC069ADBOA,SC069ADELL,SC069ADVIC,SC069BDFRI,SC069BDJEF,SC07

In [13]:
final_join = final_join.fillna(0)

In [14]:
final_join.drop(["DISSOLVE_ALPHA"], axis = 1, inplace = True)

In [15]:
final_join["UNIQUE_ID"] = final_join["GEOID20"]

In [16]:
final_join.rename(columns = {'COUNTYFP20':'COUNTYFP'}, inplace = True)

In [25]:
counties = pd.read_csv("./raw-from-source/FIPS/US_FIPS_Codes.csv", dtype =str)

In [26]:
la_counties = counties[counties["State"]=="Louisiana"]

In [30]:
la_counties_dict = dict(zip(la_counties["FIPS County"], la_counties["County Name"]))

In [33]:
final_join["Parish"] = final_join["COUNTYFP"].map(la_counties_dict).fillna("MISSING")

In [34]:
final_join["Parish"].unique()

array(['Webster', 'West Carroll', 'Union', 'West Feliciana', 'Bienville',
       'St Tammany', 'La Salle', 'Allen', 'Winn', 'Ouachita', 'St Helena',
       'Ascension', 'East Baton Rouge', 'Lafayette', 'Orleans',
       'Calcasieu', 'Iberia', 'Concordia', 'Caldwell', 'Jefferson',
       'De Soto', 'Avoyelles', 'St Mary', 'Jefferson Davis', 'Terrebonne',
       'Catahoula', 'Bossier', 'Livingston', 'Jackson', 'Sabine',
       'Lafourche', 'Franklin', 'Madison', 'Lincoln', 'Richland',
       'Tangipahoa', 'Acadia', 'East Carroll', 'Red River', 'Evangeline',
       'Vermilion', 'Beauregard', 'St Landry', 'Tensas', 'Claiborne',
       'Assumption', 'West Baton Rouge', 'Cameron', 'Grant', 'St Charles',
       'St Martin', 'East Feliciana', 'Pointe Coupee', 'Washington',
       'St James', 'Natchitoches', 'Plaquemines', 'Rapides', 'St Bernard',
       'St John The Baptist', 'Morehouse', 'Vernon', 'Iberville', 'Caddo'],
      dtype=object)

In [35]:
final_join = final_join[["UNIQUE_ID", 'COUNTYFP', "Parish", 'VTDST20', 'GEOID20', 'TOT_POP', 'TOT_WHITE', 'TOT_BLACK', 'TOT_ASIAN', 'TOT_AMIND',
       'TOT_OTHER', 'TOT_HISPAN', 'VAP_TOTAL', 'VAP_WHITE', 'VAP_BLACK',
       'VAP_ASIAN', 'VAP_AMIND', 'VAP_OTHER', 'VAP_HISPAN'] + races +["geometry"]]

In [36]:
for race in races: 
    final_join[race] = final_join[race].astype(int)

In [37]:
final_join.to_file("./la_2024_prim_prec_shp/la_2024_prim_prec_shp.shp")

/Users/peterhorton/opt/anaconda3/envs/run_maup/lib/python3.8/site-packages/geopandas/io/file.py:299: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


In [39]:
import shutil

shutil.make_archive("./la_2024_prim_prec_shp", "zip", "./la_2024_prim_prec_shp")

'/Users/peterhorton/Documents/RDH/pber_local/LA_2024/la_2024_prim_prec_shp.zip'